# HVAC Demand analysis and prediction

## 0 EDA and data preparation


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)


In [3]:
df = pd.read_csv("./data/Load_data_new.csv")


In [6]:
df

,Time,air_pressure[mmHg],air_temperature[degree celcius],relative_humidity[%],wind_speed[M/S],solar_irridiation[W/m²],total_cloud_cover[from ten],electricity_demand_values[kw],heat_demand_values[kw]
0,2010-12-01 00:00:00,729.7,25.0,85.0,5.0,0,no clouds,289.567557,432.940036
1,2010-12-01 01:00:00,729.4,27.8,77.0,7.0,0,no clouds,260.168520,473.935901
2,2010-12-01 02:00:00,728.9,33.3,62.0,7.0,0,2/10–3/10.,247.273585,483.278761
3,2010-12-01 03:00:00,731.6,32.2,62.0,2.0,0,5/10.,257.955878,545.921252
4,2010-12-01 04:00:00,732.6,22.8,96.0,3.0,0,2/10–3/10.,258.255081,550.526112
...,...,...,...,...,...,...,...,...,...
70075,2018-11-28 19:00:00,733.3,24.4,60.0,3.0,262,no clouds,379.637300,626.192823
70076,2018-11-28 20:00:00,733.6,27.8,56.0,4.0,0,no clouds,369.976634,609.519358
70077,2018-11-28 21:00:00,732.1,38.3,22.0,0.0,0,no clouds,365.009491,571.465130
70078,2018-11-28 22:00:00,735.3,36.7,25.0,4.0,0,no clouds,396.966494,583.703242


In [4]:
df.head()


,Time,air_pressure[mmHg],air_temperature[degree celcius],relative_humidity[%],wind_speed[M/S],solar_irridiation[W/m²],total_cloud_cover[from ten],electricity_demand_values[kw],heat_demand_values[kw]
0,2010-12-01 00:00:00,729.7,25.0,85.0,5.0,0,no clouds,289.567557,432.940036
1,2010-12-01 01:00:00,729.4,27.8,77.0,7.0,0,no clouds,260.168520,473.935901
2,2010-12-01 02:00:00,728.9,33.3,62.0,7.0,0,2/10–3/10.,247.273585,483.278761
3,2010-12-01 03:00:00,731.6,32.2,62.0,2.0,0,5/10.,257.955878,545.921252
4,2010-12-01 04:00:00,732.6,22.8,96.0,3.0,0,2/10–3/10.,258.255081,550.526112


In [5]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70080 entries, 0 to 70079
Data columns (total 9 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Time                             70080 non-null  object 
 1   air_pressure[mmHg]               69934 non-null  float64
 2   air_temperature[degree celcius]  69903 non-null  float64
 3   relative_humidity[%]             69903 non-null  float64
 4   wind_speed[M/S]                  69125 non-null  float64
 5   solar_irridiation[W/m²]          70080 non-null  int64  
 6   total_cloud_cover[from ten]      69837 non-null  object 
 7   electricity_demand_values[kw]    70073 non-null  float64
 8   heat_demand_values[kw]           70073 non-null  float64
dtypes: float64(6), int64(1), object(2)
memory usage: 4.8+ MB


In [6]:
df.columns = [
    "Time",
    "air_pressure",
    "air_temperature",
    "relative_humidity",
    "wind_speed",
    "solar_irridiation",
    "total_cloud_cover",
    "electricity_demand_values",
    "heat_demand_values",
]


In [7]:
df.describe().T


,count,mean,std,min,25%,50%,75%,max
air_pressure,69934.0,734.588143,5.011322,716.500000,731.400000,734.200000,737.500000,757.500000
air_temperature,69903.0,17.871834,10.683280,-14.400000,10.000000,18.900000,25.600000,43.300000
relative_humidity,69903.0,60.644178,22.007274,4.000000,43.000000,61.000000,79.000000,100.000000
wind_speed,69125.0,4.828268,2.598960,0.000000,3.000000,5.000000,6.000000,26.000000
solar_irridiation,70080.0,257.293094,258.725788,0.000000,0.000000,299.500000,499.000000,699.000000
electricity_demand_values,70073.0,393.888975,239.189061,112.947618,227.707914,323.093703,476.911512,1592.893206
heat_demand_values,70073.0,263.506355,314.704564,0.000000,0.000000,137.281603,448.289876,1529.168786


In [8]:
df.duplicated().sum()


0

In [9]:
df.isnull().sum()


Time                           0
air_pressure                 146
air_temperature              177
relative_humidity            177
wind_speed                   955
solar_irridiation              0
total_cloud_cover            243
electricity_demand_values      7
heat_demand_values             7
dtype: int64

In [10]:
df["total_cloud_cover"].value_counts()


total_cloud_cover
no clouds                                                     35431
10/10.                                                        10647
7/10 – 8/10.                                                   9172
2/10–3/10.                                                     8132
5/10.                                                          6048
Sky obscured by fog and/or other meteorological phenomena.      314
4/10.                                                            93
Name: count, dtype: int64

In [11]:
df["total_cloud_cover"].unique()


array(['no clouds', '2/10–3/10.', '5/10.', '10/10.',
       'Sky obscured by fog and/or other meteorological phenomena.',
       '7/10 – 8/10.', nan, '4/10.'], dtype=object)

In [12]:
def cloud_processing(value):
    if value in [
        "no clouds",
        "Sky obscured by fog and/or other meteorological phenomena.",
    ]:
        return 0
    if value == "2/10–3/10.":
        return 0.25
    if value == "4/10.":
        return 0.4
    if value == "5/10.":
        return 0.5
    if value == "7/10 – 8/10.":
        return 0.75
    if value == "10/10.":
        return 1


In [13]:
df["total_cloud_cover_percent"] = df["total_cloud_cover"].apply(cloud_processing)


In [14]:
df["total_cloud_cover_percent"].fillna(
    df["total_cloud_cover_percent"].mean(), inplace=True
)


In [15]:
df["total_cloud_cover_percent"].isna().sum()


0

In [16]:
df.drop(columns=["total_cloud_cover"], inplace=True)


In [17]:
df["total_cloud_cover_percent"].value_counts()


total_cloud_cover_percent
0.0000    35745
1.0000    10647
0.7500     9172
0.2500     8132
0.5000     6048
0.3239      243
0.4000       93
Name: count, dtype: int64

In [18]:
df


,Time,air_pressure,air_temperature,relative_humidity,wind_speed,solar_irridiation,electricity_demand_values,heat_demand_values,total_cloud_cover_percent
0,2010-12-01 00:00:00,729.7,25.0,85.0,5.0,0,289.567557,432.940036,0.00
1,2010-12-01 01:00:00,729.4,27.8,77.0,7.0,0,260.168520,473.935901,0.00
2,2010-12-01 02:00:00,728.9,33.3,62.0,7.0,0,247.273585,483.278761,0.25
3,2010-12-01 03:00:00,731.6,32.2,62.0,2.0,0,257.955878,545.921252,0.50
4,2010-12-01 04:00:00,732.6,22.8,96.0,3.0,0,258.255081,550.526112,0.25
...,...,...,...,...,...,...,...,...,...
70075,2018-11-28 19:00:00,733.3,24.4,60.0,3.0,262,379.637300,626.192823,0.00
70076,2018-11-28 20:00:00,733.6,27.8,56.0,4.0,0,369.976634,609.519358,0.00
70077,2018-11-28 21:00:00,732.1,38.3,22.0,0.0,0,365.009491,571.465130,0.00
70078,2018-11-28 22:00:00,735.3,36.7,25.0,4.0,0,396.966494,583.703242,0.00


In [19]:
df["Time"] = pd.to_datetime(df["Time"])
df.set_index("Time",inplace=True)


In [20]:
df.query(" Time == '2018-01-01'")


,air_pressure,air_temperature,relative_humidity,wind_speed,solar_irridiation,electricity_demand_values,heat_demand_values,total_cloud_cover_percent
Time,,,,,,,,
2018-01-01,731.8,23.9,64.0,1.0,0,535.388601,621.766744,0.0


In [ ]:
for _ in ["air_pressure", "air_temperature", "relative_humidity", "wind_speed"]:
    df[_].fillna(df[_].median(), inplace=True)


In [ ]:
df.isnull().sum()


Time                         0
air_pressure                 0
air_temperature              0
relative_humidity            0
wind_speed                   0
solar_irridiation            0
electricity_demand_values    7
heat_demand_values           7
total_cloud_cover_percent    0
dtype: int64

In [ ]:
df[["electricity_demand_values", "heat_demand_values"]] = df[
    ["electricity_demand_values", "heat_demand_values"]
].ffill()


In [ ]:
df.isnull().sum()


Time                         0
air_pressure                 0
air_temperature              0
relative_humidity            0
wind_speed                   0
solar_irridiation            0
electricity_demand_values    0
heat_demand_values           0
total_cloud_cover_percent    0
dtype: int64

In [ ]:
df.columns


Index(['Time', 'air_pressure', 'air_temperature', 'relative_humidity',
       'wind_speed', 'solar_irridiation', 'electricity_demand_values',
       'heat_demand_values', 'total_cloud_cover_percent'],
      dtype='object')

In [ ]:
df = df[['Time', 'air_pressure', 'air_temperature', 'relative_humidity',
       'wind_speed', 'solar_irridiation','total_cloud_cover_percent', 'electricity_demand_values',
       'heat_demand_values']]


In [ ]:
df


,Time,air_pressure,air_temperature,relative_humidity,wind_speed,solar_irridiation,total_cloud_cover_percent,electricity_demand_values,heat_demand_values
0,2010-12-01 00:00:00,729.7,25.0,85.0,5.0,0,0.00,289.567557,432.940036
1,2010-12-01 01:00:00,729.4,27.8,77.0,7.0,0,0.00,260.168520,473.935901
2,2010-12-01 02:00:00,728.9,33.3,62.0,7.0,0,0.25,247.273585,483.278761
3,2010-12-01 03:00:00,731.6,32.2,62.0,2.0,0,0.50,257.955878,545.921252
4,2010-12-01 04:00:00,732.6,22.8,96.0,3.0,0,0.25,258.255081,550.526112
...,...,...,...,...,...,...,...,...,...
70075,2018-11-28 19:00:00,733.3,24.4,60.0,3.0,262,0.00,379.637300,626.192823
70076,2018-11-28 20:00:00,733.6,27.8,56.0,4.0,0,0.00,369.976634,609.519358
70077,2018-11-28 21:00:00,732.1,38.3,22.0,0.0,0,0.00,365.009491,571.465130
70078,2018-11-28 22:00:00,735.3,36.7,25.0,4.0,0,0.00,396.966494,583.703242


In [ ]:
# df['Time'] = pd.to_datetime(df['Time'])


In [ ]:
df.describe().T


,count,mean,std,min,25%,50%,75%,max
air_pressure,70080.0,734.587334,5.006131,716.500000,731.400000,734.200000,737.500000,757.500000
air_temperature,70080.0,17.874431,10.669905,-14.400000,10.000000,18.900000,25.600000,43.300000
relative_humidity,70080.0,60.645077,21.979472,4.000000,43.000000,61.000000,79.000000,100.000000
wind_speed,70080.0,4.830608,2.581267,0.000000,3.000000,5.000000,6.000000,26.000000
solar_irridiation,70080.0,257.293094,258.725788,0.000000,0.000000,299.500000,499.000000,699.000000
total_cloud_cover_percent,70080.0,0.323900,0.387351,0.000000,0.000000,0.000000,0.750000,1.000000
electricity_demand_values,70080.0,393.894392,239.177728,112.947618,227.708651,323.095501,476.911512,1592.893206
heat_demand_values,70080.0,263.551003,314.720551,0.000000,0.000000,137.430266,448.333458,1529.168786


In [ ]:
# df.to_csv("./data/Load_data_01.csv", index=False)
